# SFT: Step 01: Download Gemma 4 E4B

**Goal**: Fetch Gemma 4 E4B (instruction-tuned) weights into Google Drive so they persist across Colab sessions.

**Prereqs (one-time)**:
1. Runtime type: **A100 80GB** (Runtime → Change runtime type → GPU → A100, High RAM).
2. Add HF token to Colab Secrets: left sidebar → Secrets (🔑) → `HF_TOKEN` = `hf_xxx…` with read access.
3. Mount on to Google Drive.

In [ ]:
# --- 1. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- 2. Install deps (huggingface_hub is usually preinstalled; pin fresh versions anyway) ---
!pip install -q -U huggingface_hub hf_transfer

In [ ]:
# --- 3. Configuration ---
import os
from pathlib import Path

MODEL_ID = 'google/gemma-4-e4b-it'        # Gemma 4 Efficient 4B, instruction-tuned
DRIVE_CACHE = Path('/content/drive/MyDrive/hf_cache')  # persistent across sessions
MODEL_DIR   = DRIVE_CACHE / 'models' / MODEL_ID.replace('/', '__')

DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Route HF cache to Drive so repeated runs don't re-download
os.environ['HF_HOME']       = str(DRIVE_CACHE)
os.environ['HF_HUB_CACHE']  = str(DRIVE_CACHE / 'hub')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # fast parallel downloads

print(f'Model ID:     {MODEL_ID}')
print(f'Drive cache:  {DRIVE_CACHE}')
print(f'Model path:   {MODEL_DIR}')

In [ ]:
# --- 4. HF auth from Colab Secrets ---
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets (left sidebar → 🔑) before running this cell.'
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face.')

In [ ]:
# --- 5. Download model snapshot into the Drive-backed cache ---
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(MODEL_DIR),
    local_dir_use_symlinks=False,        # copy files into Drive (symlinks don't survive Drive syncing)
    resume_download=True,                 # safe to re-run if interrupted
)
print(f'Snapshot saved to: {local_path}')
!du -sh "$local_path"

In [ ]:
# --- 6. Verify: load the tokenizer + inspect model config (no full weight load yet) ---
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
config    = AutoConfig.from_pretrained(str(MODEL_DIR))

print(f'Tokenizer vocab size: {tokenizer.vocab_size}')
print(f'Model type:           {config.model_type}')
print(f'Hidden size:          {config.hidden_size}')
print(f'Num layers:           {getattr(config, "num_hidden_layers", None)}')
print(f'Num attention heads:  {getattr(config, "num_attention_heads", None)}')

## Next steps (separate notebooks)

- `02_prepare_sft_dataset.ipynb` — shape v3 transactions into SFT chat-template JSONL
- `03_train_lora.ipynb` — LoRA / QLoRA fine-tuning on the A100
- `04_eval_judge.ipynb` — held-out evaluation

None of those exist yet — ask when you're ready.